# CSET343 – AI in Healthcare | Lab Assignment 6
## CNN-based Chest X-Ray Classification: Normal vs Pneumonia

**Objective:** Train and evaluate a CNN to classify chest X-ray images as `NORMAL` or `PNEUMONIA`.

> **Dataset note:** This notebook uses the commonly used Kaggle Chest X-Ray Images (Pneumonia) dataset. Download `chest_xray.zip` from Kaggle and upload it when prompted, or place it in your Google Drive. This dataset is for an educational classification exercise, not a clinical diagnosis tool.

**What this notebook does**
1. Loads the dataset and displays 5 random images from each class.
2. Reports class counts.
3. Resizes images to 128×128 grayscale.
4. Extracts HOG, LBP and GLCM features for a manageable sample.
5. Splits data into 70% train, 15% validation and 15% test.
6. Trains a baseline CNN and evaluates accuracy, precision, recall, F1, AUROC and a confusion matrix.
7. Includes a simple validation-based hyperparameter comparison.


## 1. Install required libraries

In [ ]:
!pip -q install scikit-image scikit-learn seaborn

## 2. Upload and extract the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select chest_xray.zip

import os, zipfile
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError("Please upload the dataset ZIP file, e.g. chest_xray.zip")

extract_dir = "/content/chest_xray_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

# Find the directory that contains train/test/val folders
DATA_DIR = None
for root, dirs, files_in_dir in os.walk(extract_dir):
    if {'train', 'test'}.issubset(set(dirs)):
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find folders named train and test. "
        "Check that you uploaded the original chest_xray ZIP."
    )

print("Dataset directory:", DATA_DIR)
print("Available folders:", os.listdir(DATA_DIR))

## 3. Task 1 — Explore class counts and display sample images

In [ ]:
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
TRAIN_DIR = os.path.join(DATA_DIR, "train")

def image_files(folder):
    return [p for p in Path(folder).rglob("*")
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

train_counts = {c: len(image_files(os.path.join(TRAIN_DIR, c))) for c in CLASS_NAMES}
display(pd.DataFrame({"Class": list(train_counts.keys()),
                      "Number of images (train folder)": list(train_counts.values())}))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, cls in enumerate(CLASS_NAMES):
    paths = image_files(os.path.join(TRAIN_DIR, cls))
    if not paths:
        print(f"No images found for {cls}")
        continue
    chosen = random.sample(paths, min(5, len(paths)))
    for col, ax in enumerate(axes[row]):
        ax.axis("off")
        if col < len(chosen):
            ax.imshow(Image.open(chosen[col]).convert("L"), cmap="gray")
            ax.set_title(cls)
plt.tight_layout()
plt.show()

## 4. Build image paths and preprocessing

The original dataset usually contains `train`, `val`, and `test` folders. To obtain a reproducible 70/15/15 split, this notebook combines the images and makes a stratified split. Images are converted to grayscale and resized to 128×128.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

all_paths, all_labels = [], []
for cls, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    paths = image_files(os.path.join(DATA_DIR, cls))  # in case class folders are at root
    if not paths:
        paths = []
        for split in ["train", "val", "test"]:
            paths.extend(image_files(os.path.join(DATA_DIR, split, cls)))
    all_paths.extend(paths)
    all_labels.extend([label] * len(paths))

# Remove duplicate file paths while preserving order
pairs = list(dict.fromkeys(zip(map(str, all_paths), all_labels)))
all_paths = np.array([p for p, y in pairs])
all_labels = np.array([y for p, y in pairs], dtype=np.int64)

print("Total images:", len(all_paths))
print("Normal:", int((all_labels == 0).sum()))
print("Pneumonia:", int((all_labels == 1).sum()))

# 70% train, 15% validation, 15% test
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(y_train)} ({len(y_train)/len(all_labels):.1%})")
print(f"Validation: {len(y_val)} ({len(y_val)/len(all_labels):.1%})")
print(f"Test: {len(y_test)} ({len(y_test)/len(all_labels):.1%})")

In [ ]:
IMG_SIZE = 128

def load_image(path):
    img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32) / 255.0

# Load all images into memory. If RAM is limited, reduce IMG_SIZE or use a smaller subset.
X_train = np.stack([load_image(p) for p in X_train_paths])[..., np.newaxis]
X_val   = np.stack([load_image(p) for p in X_val_paths])[..., np.newaxis]
X_test  = np.stack([load_image(p) for p in X_test_paths])[..., np.newaxis]

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

## 5. Task 2 — HOG, LBP and GLCM feature extraction

These handcrafted features are computed for a **small sample** to keep Colab runtime reasonable. The CNN below learns features directly from pixels. HOG, LBP and GLCM are included to demonstrate the requested preprocessing methods.

In [ ]:
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from skimage.util import img_as_ubyte

def extract_features(paths, max_images=300):
    paths = list(paths[:max_images])
    rows = []
    for path in paths:
        gray = img_as_ubyte(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE)))

        # HOG descriptor
        hog_vec = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                      cells_per_block=(2, 2), block_norm="L2-Hys")

        # LBP histogram
        lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11),
                                   range=(0, 10), density=True)

        # GLCM properties
        glcm = graycomatrix(gray, distances=[1], angles=[0],
                            levels=256, symmetric=True, normed=True)
        glcm_vec = np.array([
            graycoprops(glcm, "contrast")[0, 0],
            graycoprops(glcm, "dissimilarity")[0, 0],
            graycoprops(glcm, "homogeneity")[0, 0],
            graycoprops(glcm, "energy")[0, 0],
            graycoprops(glcm, "correlation")[0, 0],
            graycoprops(glcm, "ASM")[0, 0]
        ])
        rows.append(np.concatenate([hog_vec, lbp_hist, glcm_vec]))
    return np.asarray(rows)

# Use a balanced small sample for feature extraction demonstration
rng = np.random.default_rng(42)
feature_paths, feature_labels = [], []
for label in [0, 1]:
    indices = np.where(y_train == label)[0]
    chosen = rng.choice(indices, size=min(150, len(indices)), replace=False)
    feature_paths.extend(X_train_paths[chosen])
    feature_labels.extend(y_train[chosen])

feature_matrix = extract_features(feature_paths)
feature_labels = np.asarray(feature_labels)
print("Feature matrix shape:", feature_matrix.shape)
print("Each vector contains HOG + LBP histogram + GLCM properties.")

## 6. Task 3 — Train a baseline CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

def build_cnn(filters=32, dropout=0.3, learning_rate=1e-3):
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.Conv2D(filters, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(filters * 2, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(filters * 4, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

class_weights_values = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weights = {i: w for i, w in enumerate(class_weights_values)}
print("Class weights:", class_weights)

model = build_cnn()
model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.show()

## 7. Hyperparameter tuning (simple validation comparison)

For a student-friendly alternative to expensive k-fold cross-validation on thousands of images, compare a few CNN configurations using the validation set. Choose the best configuration by validation AUROC. For a formal report, mention that this is a validation-based search rather than k-fold cross-validation.

In [ ]:
configs = [
    {"filters": 16, "dropout": 0.3, "learning_rate": 1e-3},
    {"filters": 32, "dropout": 0.3, "learning_rate": 1e-3},
    {"filters": 32, "dropout": 0.5, "learning_rate": 3e-4},
]

tuning_results = []
for cfg in configs:
    print("Testing:", cfg)
    trial = build_cnn(**cfg)
    trial.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=4, batch_size=32,
        class_weight=class_weights,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        )],
        verbose=0
    )
    val_loss, val_acc, val_auc = trial.evaluate(X_val, y_val, verbose=0)
    tuning_results.append({**cfg, "val_accuracy": val_acc, "val_AUROC": val_auc})
    del trial
    tf.keras.backend.clear_session()

tuning_df = pd.DataFrame(tuning_results).sort_values("val_AUROC", ascending=False)
display(tuning_df)
best_cfg = tuning_df.iloc[0]
print("Best configuration by validation AUROC:", best_cfg.to_dict())

## 8. Final evaluation on the test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# Evaluate the selected configuration on the test set
best_model = build_cnn(
    filters=int(best_cfg["filters"]),
    dropout=float(best_cfg["dropout"]),
    learning_rate=float(best_cfg["learning_rate"])
)
best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10, batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

y_prob = best_model.predict(X_test, batch_size=32).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "AUROC": roc_auc_score(y_test, y_prob)
}
display(pd.DataFrame([metrics]).round(4))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues", values_format="d")
plt.title("CNN Confusion Matrix")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True Normal (TN):", tn)
print("Normal images predicted as Pneumonia (FP):", fp)
print("Pneumonia images predicted as Normal (FN):", fn)
print("True Pneumonia (TP):", tp)

print("\nInterpretation:")
print("- False positives (FP): normal X-rays incorrectly predicted as pneumonia.")
print("- False negatives (FN): pneumonia X-rays incorrectly predicted as normal.")
print("- False negatives are especially important to review in a healthcare screening task.")

## 9. Final result and conclusion

Use the actual values printed in the evaluation table in your assignment report. Do not copy made-up scores.

**Conclusion template:**  
“A CNN was trained to classify chest X-ray images into Normal and Pneumonia classes. Images were resized to 128×128 grayscale. HOG, LBP and GLCM feature extraction was demonstrated, and the CNN was evaluated using accuracy, precision, recall, F1-score and AUROC. The confusion matrix showed the number of correct and incorrect predictions. The best hyperparameter configuration was selected using validation AUROC. This model is for academic use only and must not be used for real medical diagnosis.”